# Generate reports for disaster monitoring

This pipeline creates disaster reports via LLMs, following the methods introduced by [Cantini et al. (2025)](https://www.sciencedirect.com/science/article/pii/S246869642400020X).

In [ ]:
import pandas as pd
import pathlib

def read_tsv(filepath):
    """Read a TSV file and return its contents."""
    import pandas as pd
    return pd.read_csv(filepath, sep='\t')


def create_report(row):
    """Create a report for a given row."""
    report = "title: " + row["title"] + "\n\n" + "introduction: " + row["introduction"] + "\n\n" + "content: " + row["content"]
    return report


save_path = pathlib.Path("../exp/disaster_report_writer")

paths = {
    "gemini_2.5_flash": save_path / "gemini_2.5_flash" / "reports.tsv",
    "gpt_4o_mini_2024_07_18": save_path / "gpt_4o_mini_2024_07_18" / "reports.tsv",
}

data = {}

for model, path in paths.items():
    df = read_tsv(path)
    df["report"] = df.apply(create_report, axis=1)
    data[model] = {"df": df, "path": path}
    print(f"Loaded {model} with {len(df)} rows")

In [ ]:
print(data["gpt_4o_mini_2024_07_18"]["df"].iloc[10]["report"])

In [ ]:
# Merge all dataframes on city and state columns
# Start with the first dataframe
model_names = list(data.keys())
merged_df = data[model_names[0]]["df"].copy()

# rename columns to include model name (except city and state columns)
cols_to_rename = [col for col in merged_df.columns if col not in ["city", "state"]]
merged_df = merged_df.rename(columns={col: f"{col}_{model_names[0]}" for col in cols_to_rename})



# Merge with remaining dataframes
for model in model_names[1:]:
    df = data[model]["df"].copy()

    # Rename columns to include model name (except city and state columns)
    cols_to_rename = [col for col in df.columns if col not in ["city", "state"]]
    df = df.rename(columns={col: f"{col}_{model}" for col in cols_to_rename})

    # Merge on city and state and track dropped rows
    before_merge = len(merged_df)
    merged_df = merged_df.merge(df, on=["city", "state"], how="inner", suffixes=("", f"_{model}"))
    after_merge = len(merged_df)

    if before_merge != after_merge:
        print(f"Warning: {before_merge - after_merge} rows dropped when merging with {model}")
        # Find which city/state combinations were dropped
        merged_locations = set(zip(merged_df["city"], merged_df["state"]))
        df_locations = set(zip(df["city"], df["state"]))
        dropped_locations = merged_locations - df_locations
        if dropped_locations:
            print(f"  Dropped city/state combinations: {dropped_locations}")



print(f"Merged dataframe shape: {merged_df.shape}")
print(f"len(merged_df): {len(merged_df)}")
print(f"Columns: {merged_df.columns.tolist()}")
merged_df.head()


In [ ]:
df_to_save = merged_df.copy()
df_to_save["model_a"] = "gemini_2.5_flash"
df_to_save["model_b"] = "gpt_4o_mini_2024_07_18"
df_to_save["text_a"] = df_to_save["report_gemini_2.5_flash"]
df_to_save["text_b"] = df_to_save["report_gpt_4o_mini_2024_07_18"]

# remove other columns
df_to_save = df_to_save[["model_a", "model_b", "text_a", "text_b"]]

# save to csv
df_to_save.to_csv(save_path / "pairwise_reports.csv", index=False)

